print

In [19]:
print("Current directory content:")

Current directory content:


In [20]:
import os
repo_id = "google-colab-vscode-workflow"


try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("="*60)
    print("Running in Google Colab")
    os.chdir("/content")
    if os.path.isdir(repo_id):
        %cd {repo_id}
        !git pull
    else:
        !git clone https://github.com/sagecodes/{repo_id} #repo to work in Colab
        %cd {repo_id}
        !uv pip install -r requirements.txt
else:
    print("Not running in Google Colab")

Running in Google Colab
/content/google-colab-vscode-workflow
Already up to date.


In [21]:
!ls

colab-vscode-workflow.ipynb  iris_dataset.csv  requirements.txt  test.py
downloaded_iris_dataset.csv  README.md	       test2.py


In [22]:
!pwd

/content/google-colab-vscode-workflow


saving files hf

In [23]:
import pandas as pd
from sklearn.datasets import load_iris

# Load the iris dataset
iris = load_iris()

# Create a DataFrame with the features and target
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['target'] = iris.target
df['target_name'] = df['target'].map({0: iris.target_names[0], 1: iris.target_names[1], 2: iris.target_names[2]})

# Save to CSV
df.to_csv('iris_dataset.csv', index=False)

print("Iris dataset saved as 'iris_dataset.csv'")
print(f"Dataset shape: {df.shape}")
print("\nFirst few rows:")
print(df.head())

Iris dataset saved as 'iris_dataset.csv'
Dataset shape: (150, 6)

First few rows:
   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  \
0                5.1               3.5                1.4               0.2   
1                4.9               3.0                1.4               0.2   
2                4.7               3.2                1.3               0.2   
3                4.6               3.1                1.5               0.2   
4                5.0               3.6                1.4               0.2   

   target target_name  
0       0      setosa  
1       0      setosa  
2       0      setosa  
3       0      setosa  
4       0      setosa  


In [24]:
!ls

colab-vscode-workflow.ipynb  iris_dataset.csv  requirements.txt  test.py
downloaded_iris_dataset.csv  README.md	       test2.py


In [25]:
import os
from dotenv import load_dotenv
from huggingface_hub import HfApi
import getpass

# Load environment variables from .env file
load_dotenv()

def upload_file_to_hf(file_path: str, repo_name: str, path_in_repo: str = None) -> str:
    # Check if we're in Colab
    try:
        import google.colab
        IN_COLAB = True
    except ImportError:
        IN_COLAB = False
    
    hf_token = None
    
    if IN_COLAB:
        # In Colab, just ask for input to avoid timeout issues
        print("Running in Google Colab - please enter your HF token:")
        hf_token = getpass.getpass("Hugging Face token: ")
    else:
        # Local environment - try .env file first
        hf_token = os.getenv("HF_TOKEN")
        if hf_token is None:
            print("HF_TOKEN not found in environment variables.")
            hf_token = getpass.getpass("Please enter your Hugging Face token: ")
        else:
            print("Using Hugging Face token from environment.")
    
    if not hf_token or not hf_token.strip():
        return "Error: No HF token provided"

    # Create a new repository (if it doesn't exist)
    api = HfApi()
    api.create_repo(repo_name, token=hf_token, exist_ok=True)

    # Use filename if path_in_repo not specified
    if path_in_repo is None:
        path_in_repo = os.path.basename(file_path)

    # Upload the file to the HF repository
    api.upload_file(
        path_or_fileobj=file_path,
        path_in_repo=path_in_repo,
        repo_id=repo_name,
        commit_message="Upload file",
        token=hf_token
    )
    return f"File uploaded to Hugging Face Hub: {repo_name}/{path_in_repo}"

In [26]:
# Test the upload function with iris dataset
# Make sure to set your HF_TOKEN environment variable first
# Example: export HF_TOKEN="your_token_here"

try:
    result = upload_file_to_hf(
        file_path="iris_dataset.csv", 
        repo_name="sagecodes/vs-colab-test"  # Replace with your HF username
    )
    print(result)
except Exception as e:
    print(f"Upload failed: {e}")
    print("Make sure to:")
    print("1. Set HF_TOKEN environment variable")
    print("2. Replace 'your-username' with your actual Hugging Face username")

Running in Google Colab - please enter your HF token:


No files have been modified since last commit. Skipping to prevent empty commit.


File uploaded to Hugging Face Hub: sagecodes/vs-colab-test/iris_dataset.csv


In [29]:
from huggingface_hub import hf_hub_download
import shutil
import os

# Download files from Hugging Face repository
repo_id = "sagecodes/vs-colab-test"  # Replace with the repo you uploaded to
filename = "iris_dataset.csv"
local_filename = "downloaded_iris_dataset.csv"  # Local file path

try:
    # Download to HF cache first
    cached_file = hf_hub_download(repo_id=repo_id, filename=filename)
    print(f"File downloaded to cache: {cached_file}")
    
    # Copy to proper local path
    shutil.copy2(cached_file, local_filename)
    print(f"File copied to: {os.path.abspath(local_filename)}")
    
    # Verify the download by reading the local file
    import pandas as pd
    df = pd.read_csv(local_filename)
    print(f"\nDownloaded dataset shape: {df.shape}")
    print(df.head())
    
except Exception as e:
    print(f"Download failed: {e}")
    print("Make sure the repository and file exist")

File downloaded to cache: /root/.cache/huggingface/hub/models--sagecodes--vs-colab-test/snapshots/6ab534884de5384ffabfba57679ecf744948f75d/iris_dataset.csv
File copied to: /content/google-colab-vscode-workflow/downloaded_iris_dataset.csv

Downloaded dataset shape: (150, 6)
   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  \
0                5.1               3.5                1.4               0.2   
1                4.9               3.0                1.4               0.2   
2                4.7               3.2                1.3               0.2   
3                4.6               3.1                1.5               0.2   
4                5.0               3.6                1.4               0.2   

   target target_name  
0       0      setosa  
1       0      setosa  
2       0      setosa  
3       0      setosa  
4       0      setosa  


In [30]:
!ls

colab-vscode-workflow.ipynb  iris_dataset.csv  requirements.txt  test.py
downloaded_iris_dataset.csv  README.md	       test2.py
